# S6E5 — CatBoost Baseline

Raw features only — no manual feature engineering. CatBoost handles `driver`, `compound`, and `race` natively via ordered target statistics.

Submission = average of 5 fold test predictions (no full-data retrain).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold

In [ ]:
DATA_DIR = Path("..") / "data"
SUBMISSIONS_DIR = Path("..") / "submissions"

CAT_FEATURES = ["driver", "compound", "race"]

CAT_PARAMS = {
    "iterations": 2000,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "auto_class_weights": "Balanced",
    "early_stopping_rounds": 50,
    "verbose": 200,
}

In [ ]:
def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    def clean(df):
        df = df.copy()
        df.columns = (
            df.columns.str.lower()
            .str.replace(r"\s+", "_", regex=True)
            .str.replace(r"[^a-z0-9_]", "", regex=True)
        )
        return df
    train = clean(pd.read_csv(DATA_DIR / "train.csv"))
    test  = clean(pd.read_csv(DATA_DIR / "test.csv"))
    return train, test


def run_cv(
    train: pd.DataFrame,
    test: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray]:
    """StratifiedKFold CV. Returns OOF preds and averaged test preds."""
    feature_cols = [c for c in train.columns if c not in ("id", "pitnextlap")]
    X = train[feature_cols]
    y = train["pitnextlap"]
    X_test = test[feature_cols]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds  = np.zeros(len(train))
    test_preds = np.zeros(len(test))
    fold_aucs  = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostClassifier(**CAT_PARAMS)
        model.fit(
            X_tr, y_tr,
            cat_features=CAT_FEATURES,
            eval_set=(X_val, y_val),
            use_best_model=True,
        )

        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / skf.n_splits

        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        fold_aucs.append(fold_auc)
        print(f"Fold {fold + 1} | AUC: {fold_auc:.5f} | best_iter: {model.best_iteration_}")

    oof_auc = roc_auc_score(y, oof_preds)
    print(f"\nOOF AUC: {oof_auc:.5f} \u00b1 {np.std(fold_aucs):.5f}")
    return oof_preds, test_preds


def save_submission(test: pd.DataFrame, preds: np.ndarray, oof_auc: float) -> Path:
    SUBMISSIONS_DIR.mkdir(exist_ok=True)
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    fname = SUBMISSIONS_DIR / f"submission_cat_auc{oof_auc:.4f}_{timestamp}.csv"
    pd.DataFrame({"id": test["id"], "PitNextLap": preds}).to_csv(fname, index=False)
    print(f"Saved: {fname}")
    return fname

## Load data

In [ ]:
train, test = load_data()
print(f"Train: {train.shape}  |  Test: {test.shape}")
print(f"Target rate: {train['pitnextlap'].mean():.3f}")
train.head(3)

## 5-Fold Stratified CV

In [ ]:
oof_preds, test_preds = run_cv(train, test)

In [ ]:
oof_auc = roc_auc_score(train["pitnextlap"], oof_preds)
fpr, tpr, _ = roc_curve(train["pitnextlap"], oof_preds)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"CatBoost OOF (AUC={oof_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("OOF ROC Curve — CatBoost baseline")
plt.legend()
plt.tight_layout()
plt.show()

## Submission

Test predictions are the average of the 5 fold models — no full-data retrain.

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(test_preds, bins=50, edgecolor="white")
plt.xlabel("Predicted probability")
plt.ylabel("Count")
plt.title("Test prediction distribution")
plt.tight_layout()
plt.show()
print(f"Test pred mean: {test_preds.mean():.4f}  |  Train target mean: {train['pitnextlap'].mean():.4f}")

In [ ]:
submission_path = save_submission(test, test_preds, oof_auc)
print(f"Submission saved: {submission_path}")